In [ ]:
import numpy as np
import numpy as np
import torch
import sys, os
sys.path.append(os.path.abspath(".."))  # project root
print(os.path.abspath(".."))
from datasets.data import SpatialDataset
from datasets.transformerRegressorDataClass import TransformerPointDataset, collate_point_batches
from torch.utils.data import DataLoader

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt
import numpy as np

def degrees_to_meters(degree_res, latitude):
    """Convert resolution from degrees to meters."""
    # 1 degree latitude ≈ 111,320 meters
    lat_res_m = degree_res * 111320  
    
    # 1 degree longitude ≈ 111,320 * cos(latitude) meters
    lon_res_m = degree_res * 111320 * np.cos(np.radians(latitude))  
    
    return lat_res_m, lon_res_m  

def cropped_to_tiff(dt2_file, output_path):
    # Open and crop
    with rasterio.open(dt2_file) as src:
        window = from_bounds(min_lon, min_lat, max_lon, max_lat, src.transform)
        cropped = src.read(1, window=window)
        transform = src.window_transform(window)

        profile = src.profile.copy()
        profile.update({
            "height": cropped.shape[0],
            "width": cropped.shape[1],
            "transform": transform,
            "driver": "GTiff"
        })

    # Save as GeoTIFF
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(cropped, 1)

# -------------------- Parameters --------------------
dt2_file = "Transformer_Map_Interp/datasets/n32_e035_1arc_v3.dt2"
# -------------------- Load Full Map --------------------
with rasterio.open(dt2_file) as src:
    elevation = src.read(1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    degree_res_x, degree_res_y  = src.res  # Pixel size (degrees per pixel)
    bounds = src.bounds  # Geographic extent (min/max lon, lat)
    mid_latitude = (src.bounds.top + src.bounds.bottom) / 2

    # Convert resolution to meters
    lat_res_m, lon_res_m = degrees_to_meters(degree_res_x, mid_latitude)